In [ ]:
import sys
sys.path.append("..")

from torch.utils.data import DataLoader
from wimusim.datasets import CPM

## Introduction

This notebook demonstrates **parameter transformations** in WIMUSim to generate diverse
virtual IMU data for training and testing HAR / IMU estimation models.

### Dataset Strategy
| Role | Dataset | Why |
|---|---|---|
| **Train** | MoVi | 90 subjects × 20 activities, SMPL fits + 17 IMUs |
| **Test** | TotalCapture | Different lab setting, 13 IMUs, 5 subjects |
| **Test** | EMDB | In-the-wild outdoor sequences, 6 IMUs, moving camera |

### What We Will Do
1. **Load identified parameters** from `parameter_identification.ipynb` for MoVi subjects.
2. **CPM (train)**: Mix B/D/P/H across MoVi subjects → large virtual training set.
3. **PDG**: Fix a subject's B/P/H, vary D → subject-specific augmentation.
4. **Cross-dataset evaluation**: Load TotalCapture / EMDB params for zero-shot testing.

## 1. Load Identified Parameters (MoVi — Training Set)

After running `parameter_identification.ipynb` for each MoVi subject/activity,
save the optimized parameters as a `.pkl` file:

```python
import pickle
cpm_params = {
    "B_list":      [...],   # one WIMUSim.Body per subject/sequence
    "D_list":      [...],   # one WIMUSim.Dynamics per subject/sequence
    "P_list":      [...],   # one WIMUSim.Placement per subject/sequence
    "H_list":      [...],   # one WIMUSim.Hardware per subject/sequence
    "target_list": [...],   # real IMU dict per subject/sequence
}
with open("cpm_params_movi.pkl", "wb") as f:
    pickle.dump(cpm_params, f)
```

In [ ]:
import pickle

cpm_param_path = "cpm_params_movi.pkl"

with open(cpm_param_path, "rb") as f:
    cpm_params = pickle.load(f)

print(f"Loaded {len(cpm_params['B_list'])} MoVi sequences")

## 2. Define the Comprehensive Parameter Mixing (CPM) Object

The **CPM** object generates new combinations of B, D, P, and H parameters.
By mixing these parameter sets across subjects, we create diverse virtual IMU data
that captures realistic variability in body morphology, motion, sensor placement,
and hardware noise.

- **`window`**: Length of the sliding window for time-series segmentation.
- **`stride`**: Step size between consecutive windows.

In [ ]:
movi_cpm_dataset = CPM(
    B_list=cpm_params["B_list"],
    D_list=cpm_params["D_list"],
    P_list=cpm_params["P_list"],
    H_list=cpm_params["H_list"],
    target_list=cpm_params["target_list"],
    window=100,
    stride=25,
    acc_only=False,
)

## **3. Generate Data with CPM**

Now that we have defined the **Comprehensive Parameter Mixing (CPM)** object, we can proceed to generate a diverse set of virtual IMU data using various combinations of the **Body (B)**, **Dynamics (D)**, **Placement (P)**, and **Hardware (H)** parameters.

### **Generating Virtual IMU Data**
The `generate_data()` method of the `CPM` object randomly choose `n_combinations` from the given parameter lists to create new virtual IMU data samples. Each sample represents a unique configuration, reflecting realistic variations in the human body model, body movement dynamics, sensor placements, and sensor characteristics.

- **`n_combinations`**: This parameter specifies the number of parameter combinations to generate. For example, setting `n_combinations=100` will produce 100 different virtual IMU samples by varying the B, D, P, and H parameters across subjects.

> **Note**: The generated data is stored in the `realdisp_cpm_dataset.data` attribute, and the corresponding labels are stored in `realdisp_cpm_dataset.target`.

In [ ]:
movi_cpm_dataset.generate_data(n_combinations=10)

print(f"Generated {movi_cpm_dataset.__len__()} windows of virtual IMU data (MoVi train set).")

### **Using the Generated Data with PyTorch’s DataLoader**
The generated dataset can be used with PyTorch’s `DataLoader` to efficiently batch and shuffle the data for training machine learning models. This allows us to seamlessly integrate WIMUSim’s synthetic IMU data into model training pipelines.


### **Understanding the Output Data**
- **`data`**: Contains the generated virtual IMU signals. For each sample, the data is represented as a 3D tensor of shape `[batch_size, window_size, num_features]`.
  - `batch_size`: The number of samples in each batch (e.g., 1024).
  - `window_size`: The length of each time-series window (e.g., 100).
  - `num_features`: The number of features for each IMU signal (e.g., 12 for 6-axis IMU data with acceleration and gyroscope signals).
  
- **`target`**: Corresponding label for each sample (e.g., activity type or subject ID).
- **`idx`**: Unique identifier for each data sample, useful for tracking and debugging.


In [ ]:
data_loader = DataLoader(
    movi_cpm_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=0,
)

for data, target, idx in data_loader:
    print(f"Data shape:   {data.shape}")
    print(f"Target shape: {target.shape}")
    break

## 4. Personalized Dataset Generation (PDG)

**PDG** fixes certain parameters (e.g. body morphology, sensor placement) for a
target subject while varying others to introduce realistic intra-subject variability.

Below we fix B, P, and H to Subject `p003` and vary only D (motion dynamics),
producing a dataset tailored to that individual's body shape and IMU placement
while covering diverse movements from all subjects.

In [ ]:
subject_idx = 0  # fix B/P/H to the first MoVi subject, vary D across all subjects

movi_pdg_dataset = CPM(
    B_list=cpm_params["B_list"][subject_idx : subject_idx + 1],
    D_list=cpm_params["D_list"],
    P_list=cpm_params["P_list"][subject_idx : subject_idx + 1],
    H_list=cpm_params["H_list"][subject_idx : subject_idx + 1],
    target_list=cpm_params["target_list"],
    window=100,
    stride=25,
    acc_only=False,
)

### **Generating Personalized Data**
We can now use the `generate_data()` method again to generate virtual IMU data for this subject. Since the B, P, and H parameters are fixed, only the D parameters will vary, simulating different movement patterns for Subject 3:

In [ ]:
movi_pdg_dataset.generate_data(n_combinations=5)
print(f"Generated {movi_pdg_dataset.__len__()} personalized samples.")

## **Conclusion**

In this notebook, we explored how to use WIMUSim's **parameter transformation capabilities** to generate diverse and realistic virtual IMU datasets for Human Activity Recognition (HAR) models. 